In [1]:
!pip install -qqq "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -qqq accelerate huggingface_hub hf_transfer

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 156.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 127.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 120.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 27.0 MB/s eta 0:00:00


QWEN

In [2]:
import os
import pandas as pd
import torch
from tqdm import tqdm
from datasets import load_from_disk
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
from datetime import datetime

# ==================== CONFIGURATION ====================
data_save_path = "/content/drive/MyDrive/CSE4078_Project/dataset_splits_hf"
output_dir = "/content/drive/MyDrive/CSE4078_Project/predictions"
num_samples = 10
BATCH_SIZE = 32

# ==================== LOAD DATASET ====================
test_dataset = load_from_disk(f"{data_save_path}/test_dataset")
test_sample = test_dataset.shuffle(seed=42).select(range(num_samples))
print(f"[INFO] Test sample ready: {len(test_sample)} examples")

# ==================== LOAD BASE MODEL ====================
model_name = "Qwen/Qwen3.5-2B"
print(f"[LOADING] Base model: {model_name}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=False,
)
model = FastLanguageModel.for_inference(model)

print("[CONFIG] Applying tokenizer adjustments...")
if hasattr(tokenizer, "tokenizer"):
    tokenizer = tokenizer.tokenizer
if hasattr(tokenizer, "processor"):
    tokenizer.processor = None
if hasattr(tokenizer, "image_processor"):
    tokenizer.image_processor = None
try:
    pure_tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=True)
    tokenizer = pure_tokenizer
except Exception:
    pass
tokenizer.padding_side = "left"
model.config.use_cache = True
print("[INFO] Model and tokenizer ready")

# ==================== PREPARE PROMPTS ====================
print("[INFO] Preparing prompts...")
prompts = []
doc_titles = []
doc_specs = []
questions = []
ground_truth_answers = []  # Added list to store original answers

for example in test_sample:
    doc_title = example.get("doctor_title", "Doctor")
    doc_spec = example.get("doctor_specialty") or example.get("doctor_speciality", "General Medicine")
    question = example["question_content"]
    ground_truth = example.get("question_answer", "")  # Extract ground truth answer

    system_prompt = (
        f"You are a Turkish medical doctor with the title '{doc_title}', "
        f"specializing in {doc_spec}. "
        "Answer patient questions in Turkish language only. "
        "Provide medically accurate, clear, professional, and empathetic responses. "
        "Base your answers strictly on medical facts. "
        "LENGTH REQUIREMENT: Keep your answer concise — maximum 3-4 sentences or ~80 words. "
        "Focus on the core medical advice: diagnosis context, recommended action, and when to seek in-person care. "
        "Avoid lengthy explanations, repetitive phrases, or listing multiple rare possibilities. "
        "Write as continuous paragraphs only: no bullet points, no headers, no markdown formatting."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]
    prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
    doc_titles.append(doc_title)
    doc_specs.append(doc_spec)
    questions.append(question)
    ground_truth_answers.append(ground_truth)  # Store ground truth

# ==================== BATCHED INFERENCE ====================
results = []
total_batches = (len(prompts) + BATCH_SIZE - 1) // BATCH_SIZE

print(f"[START] Running batched generation ({total_batches} batches of size {BATCH_SIZE})...")

for i in tqdm(range(0, len(prompts), BATCH_SIZE), desc="Batching", total=total_batches):
    batch_prompts = prompts[i:i+BATCH_SIZE]

    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=2048).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=192,
            temperature=0.2,
            do_sample=True,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            use_cache=True,
        )

    input_len = inputs.input_ids.shape[1]
    for j in range(len(batch_prompts)):
        response = tokenizer.decode(outputs[j][input_len:], skip_special_tokens=True)
        response = response.replace("<think>", " ").replace("</think>", " ").strip()

        results.append({
            "doctor_title": doc_titles[i+j],
            "doctor_specialty": doc_specs[i+j],
            "question_content": questions[i+j],
            "question_answer": ground_truth_answers[i+j],  # Added ground truth between content and generated
            "generated_answer": response
        })

    torch.cuda.empty_cache()

# ==================== SAVE RESULTS ====================
os.makedirs(output_dir, exist_ok=True)
model_slug = model_name.replace("/", "_")
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = os.path.join(output_dir, f"{model_slug}_{num_samples}_predictions_{timestamp}.csv")

df_results = pd.DataFrame(results)
# Enforce column order for CSV output
column_order = ["doctor_title", "doctor_specialty", "question_content", "question_answer", "generated_answer"]
df_results = df_results[column_order]

df_results.to_csv(output_path, index=False, encoding="utf-8")

print(f"[COMPLETE] Generation finished. Results saved to: {output_path}")
display(df_results.head())

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
[INFO] Test sample ready: 10 examples
[LOADING] Base model: Qwen/Qwen3.5-2B
==((====))==  Unsloth 2026.5.2: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

...

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

[CONFIG] Applying tokenizer adjustments...
[INFO] Model and tokenizer ready
[INFO] Preparing prompts...
[START] Running batched generation (1 batches of size 32)...


Batching: 100%|██████████| 1/1 [00:48<00:00, 48.24s/it]

[COMPLETE] Generation finished. Results saved to: /content/drive/MyDrive/CSE4078_Project/predictions/Qwen_Qwen3.5-2B_10_predictions_20260509_214220.csv


,doctor_title,doctor_specialty,question_content,question_answer,generated_answer
0,Doç. Dr.,ortopedi-ve-travmatoloji,Merhaba hocam sağ ayak 4.parmakta kısalık var ...,Hastanemizde tıbbi gereklilik halinde uzatma a...,"Doç. Dr. olarak, 4. parmak kısalığı (karpometa..."
1,Op. Dr.,beyin-ve-sinir-cerrahisi,Annem 59 yaşında ve 6 yıldır bel fıtığı rahats...,Annenizin ameliyat olmasında yarar var. Benim ...,"Bu durum, L3-S1 arası ileri düzey bulging ve L..."
2,Prof. Dr.,dahiliye-ve-ic-hastaliklari-nefroloji,hocam 15 aylık kızım var sol tarafta tek taraf...,Daha çocuğun yaşı çok küçük. Bir süre sonra so...,15 aylık çocuklarda tek taraflı 2. derecedeki ...
3,Doç. Dr.,uroloji,"hocam merhaba, ben 45 yaşındayım. penis boyum ...",Zayıflarsanız düzelir.,"Doç. Dr. olarak, penis içe kaçması (epispadiya..."
4,Uzm. Dr.,dahiliye-ve-ic-hastaliklari,merhaba hocam helipak ilaçlarını kullanmaya ba...,"Ayşe hanım, Tarif ettiğiniz olay öncelikle ant...",Helipak (metronidazol) kullanımıyla dışkı yeşi...


LLAMA

In [3]:
import os
import pandas as pd
import torch
from tqdm import tqdm
from datasets import load_from_disk
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
from datetime import datetime

# ==================== CONFIGURATION ====================
data_save_path = "/content/drive/MyDrive/CSE4078_Project/dataset_splits_hf"
output_dir = "/content/drive/MyDrive/CSE4078_Project/predictions"
num_samples = 10
BATCH_SIZE = 32

# ==================== LOAD DATASET ====================
test_dataset = load_from_disk(f"{data_save_path}/test_dataset")
test_sample = test_dataset.shuffle(seed=42).select(range(num_samples))
print(f"[INFO] Test sample ready: {len(test_sample)} examples")

# ==================== LOAD BASE MODEL ====================
model_name = "meta-llama/Llama-3.2-1B-Instruct"
print(f"[LOADING] Base model: {model_name}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=False,
)
model = FastLanguageModel.for_inference(model)

# ==================== TOKENIZER SETUP FOR LLAMA 3.2 ====================
print("[CONFIG] Applying Llama 3.2 tokenizer adjustments...")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

eos_token_id = tokenizer.eos_token_id if isinstance(tokenizer.eos_token_id, int) else tokenizer.eos_token_id[0]

model.config.use_cache = True
tokenizer.padding_side = "left"
print(f"[INFO] Tokenizer configured: pad_token_id={tokenizer.pad_token_id}, eos_token_id={eos_token_id}")

# ==================== PREPARE PROMPTS ====================
print("[INFO] Preparing prompts with Llama 3.2 chat template...")
prompts = []
doc_titles = []
doc_specs = []
questions = []
ground_truth_answers = []  # Added list to store question_answer

for example in test_sample:
    doc_title = example.get("doctor_title", "Doctor")
    doc_spec = example.get("doctor_specialty") or example.get("doctor_speciality", "General Medicine")
    question = example["question_content"]
    ground_truth = example.get("question_answer", "")  # Extract ground truth answer

    system_prompt = (
        f"You are a Turkish medical doctor with the title '{doc_title}', "
        f"specializing in {doc_spec}. "
        "Answer patient questions in Turkish language only. "
        "Provide medically accurate, clear, professional, and empathetic responses. "
        "Base your answers strictly on medical facts. "
        "LENGTH REQUIREMENT: Keep your answer concise — maximum 3-4 sentences or ~80 words. "
        "Focus on the core medical advice: diagnosis context, recommended action, and when to seek in-person care. "
        "Avoid lengthy explanations, repetitive phrases, or listing multiple rare possibilities. "
        "Write as continuous paragraphs only: no bullet points, no headers, no markdown formatting."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    prompts.append(prompt_text)
    doc_titles.append(doc_title)
    doc_specs.append(doc_spec)
    questions.append(question)
    ground_truth_answers.append(ground_truth)  # Store ground truth

# ==================== BATCHED INFERENCE ====================
results = []
total_batches = (len(prompts) + BATCH_SIZE - 1) // BATCH_SIZE

print(f"[START] Running batched generation ({total_batches} batches of size {BATCH_SIZE})...")

for i in tqdm(range(0, len(prompts), BATCH_SIZE), desc="Batching", total=total_batches):
    batch_prompts = prompts[i:i+BATCH_SIZE]

    inputs = tokenizer(
        batch_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=192,
            temperature=0.2,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=eos_token_id,
            use_cache=True,
        )

    input_len = inputs.input_ids.shape[1]
    for j in range(len(batch_prompts)):
        response = tokenizer.decode(
            outputs[j][input_len:],
            skip_special_tokens=True
        ).strip()

        response = response.replace("<think>", "").replace("</think>", "").strip()

        results.append({
            "doctor_title": doc_titles[i+j],
            "doctor_specialty": doc_specs[i+j],
            "question_content": questions[i+j],
            "question_answer": ground_truth_answers[i+j],  # Added ground truth between content and generated
            "generated_answer": response
        })

    torch.cuda.empty_cache()

# ==================== SAVE RESULTS ====================
os.makedirs(output_dir, exist_ok=True)
model_slug = model_name.replace("/", "_")
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = os.path.join(output_dir, f"{model_slug}_{num_samples}_predictions_{timestamp}.csv")

df_results = pd.DataFrame(results)
# Enforce column order for CSV output
column_order = ["doctor_title", "doctor_specialty", "question_content", "question_answer", "generated_answer"]
df_results = df_results[column_order]

df_results.to_csv(output_path, index=False, encoding="utf-8")

print(f"[COMPLETE] Generation finished. Results saved to: {output_path}")
display(df_results.head())

[INFO] Test sample ready: 10 examples
[LOADING] Base model: meta-llama/Llama-3.2-1B-Instruct
==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

...

Unsloth: Will load unsloth/Llama-3.2-1B-Instruct as a legacy tokenizer.


[CONFIG] Applying Llama 3.2 tokenizer adjustments...
[INFO] Tokenizer configured: pad_token_id=128004, eos_token_id=128009
[INFO] Preparing prompts with Llama 3.2 chat template...
[START] Running batched generation (1 batches of size 32)...


Batching:   0%|          | 0/1 [00:00<?, ?it/s]Both `max_new_tokens` (=192) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_M

[COMPLETE] Generation finished. Results saved to: /content/drive/MyDrive/CSE4078_Project/predictions/meta-llama_Llama-3.2-1B-Instruct_10_predictions_20260509_214314.csv


,doctor_title,doctor_specialty,question_content,question_answer,generated_answer
0,Doç. Dr.,ortopedi-ve-travmatoloji,Merhaba hocam sağ ayak 4.parmakta kısalık var ...,Hastanemizde tıbbi gereklilik halinde uzatma a...,"Merhaba, siz de sağ ayakta olan 4. parmakta kı..."
1,Op. Dr.,beyin-ve-sinir-cerrahisi,Annem 59 yaşında ve 6 yıldır bel fıtığı rahats...,Annenizin ameliyat olmasında yarar var. Benim ...,"Annem, 59 yaşındayken bel fıtığı rahatsızlığın..."
2,Prof. Dr.,dahiliye-ve-ic-hastaliklari-nefroloji,hocam 15 aylık kızım var sol tarafta tek taraf...,Daha çocuğun yaşı çok küçük. Bir süre sonra so...,Kızınızın sol taraflı 2. derecede vücutta bir ...
3,Doç. Dr.,uroloji,"hocam merhaba, ben 45 yaşındayım. penis boyum ...",Zayıflarsanız düzelir.,"Ben de aynılsınız, 45 yaşındayken penisimin in..."
4,Uzm. Dr.,dahiliye-ve-ic-hastaliklari,merhaba hocam helipak ilaçlarını kullanmaya ba...,"Ayşe hanım, Tarif ettiğiniz olay öncelikle ant...","Merhaba, your external green color is normal. ..."


GEMMA

In [ ]:
import os
import pandas as pd
import torch
from tqdm import tqdm
from datasets import load_from_disk
from transformers import AutoProcessor, AutoModelForCausalLM
from datetime import datetime
import re # Import re for regular expressions

# ==================== CONFIGURATION ====================
data_save_path = "/content/drive/MyDrive/CSE4078_Project/dataset_splits_hf"
output_dir = "/content/drive/MyDrive/CSE4078_Project/predictions"
num_samples = 10
BATCH_SIZE = 32

# ==================== LOAD DATASET ====================
test_dataset = load_from_disk(f"{data_save_path}/test_dataset")
test_sample = test_dataset.shuffle(seed=42).select(range(num_samples))
print(f"[INFO] Test sample ready: {len(test_sample)} examples")

# ==================== LOAD MODEL ====================
model_name = "google/gemma-4-E2B-it"
print(f"[LOADING] Model: {model_name}")

# Gemma-4 uses AutoProcessor (not AutoTokenizer)
processor = AutoProcessor.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="auto"
)
model.eval()

# ==================== PROCESSOR SETUP ====================
print("[CONFIG] Configuring processor...")
processor.padding_side = "left"
pad_token_id = processor.tokenizer.pad_token_id
print(f"[INFO] Processor configured: pad_token_id={pad_token_id}")

# ==================== PREPARE PROMPTS ====================
print("[INFO] Preparing prompts with Gemma-4 chat template...")
prompts = []
doc_titles = []
doc_specs = []
questions = []
ground_truth_answers = []

for example in test_sample:
    doc_title = example.get("doctor_title", "Doctor")
    doc_spec = example.get("doctor_specialty") or example.get("doctor_speciality", "General Medicine")
    question = example["question_content"]
    ground_truth = example.get("question_answer", "")

    # Gemma-4 supports native system role
    system_prompt = (
        f"You are a Turkish medical doctor with the title '{doc_title}', "
        f"specializing in {doc_spec}. "
        "Answer patient questions in Turkish language only. "
        "Provide medically accurate, clear, professional, and empathetic responses. "
        "Base your answers strictly on medical facts. "
        "LENGTH REQUIREMENT: Keep your answer concise — maximum 3-4 sentences or ~80 words. "
        "Focus on the core medical advice: diagnosis context, recommended action, and when to seek in-person care. "
        "Avoid lengthy explanations, repetitive phrases, or listing multiple rare possibilities. "
        "Write as continuous paragraphs only: no bullet points, no headers, no markdown formatting."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]

    # Gemma-4: use enable_thinking=False for standard generation
    prompt_text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )
    prompts.append(prompt_text)
    doc_titles.append(doc_title)
    doc_specs.append(doc_spec)
    questions.append(question)
    ground_truth_answers.append(ground_truth)

# ==================== BATCHED INFERENCE ====================
results = []
total_batches = (len(prompts) + BATCH_SIZE - 1) // BATCH_SIZE

print(f"[START] Running batched generation ({total_batches} batches of size {BATCH_SIZE})...")

for i in tqdm(range(0, len(prompts), BATCH_SIZE), desc="Batching", total=total_batches):
    batch_prompts = prompts[i:i+BATCH_SIZE]

    inputs = processor(
        text=batch_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=192,
            # Gemma-4 recommended sampling parameters
            temperature=1.0,
            top_p=0.95,
            top_k=64,
            do_sample=True,
            pad_token_id=processor.tokenizer.pad_token_id,
            use_cache=True,
        )

    input_len = inputs["input_ids"].shape[1]
    for j in range(len(batch_prompts)):
        raw_response = processor.decode(
            outputs[j][input_len:],
            skip_special_tokens=False
        )

        # Use parse_response to properly extract the answer
        # processor.parse_response returns a dictionary for Gemma-4, need to access 'content' key
        parsed_output = processor.parse_response(raw_response)
        response = parsed_output['content'] if 'content' in parsed_output else raw_response

        # Remove <|turn|> and <pad> tokens using regex for robustness, then strip whitespace
        response = re.sub(r'<\|turn\|>|<pad>', '', response).strip()

        results.append({
            "doctor_title": doc_titles[i+j],
            "doctor_specialty": doc_specs[i+j],
            "question_content": questions[i+j],
            "question_answer": ground_truth_answers[i+j],
            "generated_answer": response
        })

    torch.cuda.empty_cache()

# ==================== SAVE RESULTS ====================
os.makedirs(output_dir, exist_ok=True)
model_slug = model_name.replace("/", "_")
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = os.path.join(output_dir, f"{model_slug}_{num_samples}_predictions_{timestamp}.csv")

df_results = pd.DataFrame(results)
column_order = ["doctor_title", "doctor_specialty", "question_content", "question_answer", "generated_answer"]
df_results = df_results[column_order]

df_results.to_csv(output_path, index=False, encoding="utf-8")

print(f"[COMPLETE] Generation finished. Results saved to: {output_path}")
display(df_results.head())